# Run Bi-Encoder Inference on Full Dev Set (1,942 problems)

**Tests bi-encoder alone (no verifier) with different top-k values**

**IMPORTANT: Set Runtime to GPU (T4 or A100/H100)**

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository
!git clone https://github.com/matrix-mayank/mathfish.git
%cd mathfish

In [ ]:
# 3. Install dependencies
!pip install -q torch transformers sentence-transformers datasets huggingface-hub tqdm

In [ ]:
# 4. Download FULL dev dataset (1,942 problems)
!python scripts/sample_dev.py --n 999999 --output data/processed/problems_dev_full.jsonl
!wc -l data/processed/problems_dev_full.jsonl

In [ ]:
# 5. Load trained bi-encoder: from Drive OR upload
from google.colab import drive, files
import zipfile
import os
import shutil

# Option A: Load from Google Drive (faster than uploading)
drive.mount('/content/drive')
drive_zip = "/content/drive/MyDrive/bge-base/biencoder_bge_large_15k.zip"
local_zip = "biencoder_bge_large_15k.zip"

if os.path.exists(drive_zip):
    shutil.copy(drive_zip, local_zip)
    print("✅ Copied zip from Drive")
    with zipfile.ZipFile(local_zip, 'r') as z:
        z.extractall('.')
    print("✅ Extracted")
    checkpoint_dir = "."
    model_name = "BGE-base"
    print("Checkpoint files (extracted to current dir):")
    !ls -la *.pt *.json 2>/dev/null | head -5
else:
    # Option B: Upload zip manually
    print("Drive path not found. Upload your bi-encoder zip:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            if 'bge' in filename:
                os.makedirs('outputs/biencoder_bge_large_15k/checkpoint', exist_ok=True)
                zip_ref.extractall('outputs/biencoder_bge_large_15k/checkpoint')
                checkpoint_dir = 'outputs/biencoder_bge_large_15k/checkpoint'
                model_name = 'BGE-base'
            elif 'full' in filename:
                os.makedirs('outputs/biencoder_full_15k/checkpoint', exist_ok=True)
                zip_ref.extractall('outputs/biencoder_full_15k/checkpoint')
                checkpoint_dir = 'outputs/biencoder_full_15k/checkpoint'
                model_name = 'MiniLM-L6 (full)'
            else:
                os.makedirs('outputs/biencoder_5k_gpu/checkpoint', exist_ok=True)
                zip_ref.extractall('outputs/biencoder_5k_gpu/checkpoint')
                checkpoint_dir = 'outputs/biencoder_5k_gpu/checkpoint'
                model_name = 'MiniLM-L6 (5k)'
    print(f"\n✅ Using: {model_name}")

print(f"\nCheckpoint dir: {checkpoint_dir}")

In [ ]:
# 6. Run inference with multiple top-k values
import subprocess
import json
import math

results = []

for k in [3, 5, 7, 10]:
    print(f"\n{'='*60}")
    print(f"Testing top-{k}")
    print(f"{'='*60}")
    
    output_file = f"outputs/biencoder_top{k}.jsonl"
    
    subprocess.run([
        "python", "scripts/run_inference.py",
        "--checkpoint_dir", checkpoint_dir,
        "--problems_file", "data/processed/problems_dev_full.jsonl",
        "--from_hf",
        "--output_file", output_file,
        "--top_k", str(k)
    ], check=True)
    
    result = subprocess.run([
        "python", "scripts/evaluate_alignment.py",
        "--predictions_file", output_file,
        "--gold_file", "data/processed/problems_dev_full.jsonl",
        "--from_hf"
    ], capture_output=True, text=True)
    
    metrics = json.loads(result.stdout)
    metrics['top_k'] = k
    results.append(metrics)
    
    print(f"\n📊 Results:")
    print(f"  Exact Match: {metrics['exact_match']:.4f}")
    print(f"  Micro F1: {metrics['micro_f1']:.4f}")
    print(f"  Macro F1: {metrics['macro_f1']:.4f}")
    print(f"  Recall@5: {metrics['recall@5']:.4f}")
    print(f"  Recall@20: {metrics['recall@20']:.4f}")
    if "avg_graph_distance" in metrics:
        g = metrics["avg_graph_distance"]
        print(f"  Avg graph distance: {g if not math.isnan(g) else 'n/a'}")
    if "sibling_confusion_rate" in metrics:
        print(f"  Sibling confusion rate: {metrics['sibling_confusion_rate']:.4f}")

# Summary
has_graph = any("avg_graph_distance" in r for r in results)
has_sibling = any("sibling_confusion_rate" in r for r in results)
header = f"\n{'Top-K':<8} {'Exact':<10} {'Micro F1':<12} {'Macro F1':<12} {'Recall@5':<12} {'Recall@20':<12}"
if has_graph:
    header += " {'GraphDist':<12}"
if has_sibling:
    header += " {'SiblingConf':<12}"
print(f"\n{'='*80}")
print("📋 FINAL SUMMARY")
print(f"{'='*80}")
print(header)
sep_len = 80 + (24 if has_graph else 0) + (24 if has_sibling else 0)
print("-" * sep_len)
for r in results:
    row = f"{r['top_k']:<8} {r['exact_match']:<10.4f} {r['micro_f1']:<12.4f} {r['macro_f1']:<12.4f} {r['recall@5']:<12.4f} {r['recall@20']:<12.4f}"
    if has_graph:
        g = r.get("avg_graph_distance")
        row += f" {(f'{g:.2f}' if g is not None and not math.isnan(g) else 'n/a'):<12}"
    if has_sibling:
        row += f" {r.get('sibling_confusion_rate', 0):<12.4f}"
    print(row)

best_exact = max(results, key=lambda x: x["exact_match"])
best_f1 = max(results, key=lambda x: x["micro_f1"])
print(f"\n🎯 Best Exact Match: top-{best_exact['top_k']} ({best_exact['exact_match']:.4f})")
print(f"🎯 Best Micro F1: top-{best_f1['top_k']} ({best_f1['micro_f1']:.4f})")

In [ ]:
# 7. Download results
import shutil
from google.colab import files

shutil.make_archive('biencoder_inference_results', 'zip', 'outputs')
files.download('biencoder_inference_results.zip')

print(f"\n✅ Results for {model_name} on 1,942 dev problems")